# X-SDD — Eksperimen V2 (ECA setelah bn3) (3 seed, output lengkap untuk analisis)
V2: ECA di tiap bottleneck, setelah bn3 dan sebelum penjumlahan residual (posisi standar ECA-ResNet), 16 modul, +86 parameter.
Pipeline: split CSV tetap (stratified 70/30), from scratch, resize 224, normalisasi ImageNet, tanpa augmentasi,
Adam lr 1e-4, batch 10, 100 epoch, tanpa early stopping. Model yang dianalisis = bobot epoch 100.
Tanpa wandb. Semua output ditulis ke folder `exp_<varian>` di /kaggle/working dan dikemas menjadi satu zip.
**Cara mengunduh:** setelah Save & Run All (Commit) selesai → tab Output → unduh file zip `exp_<varian>.zip`.
Input yang dibutuhkan: dataset gambar X-SDD dan dataset split CSV. GPU aktif. Internet tidak diperlukan.
Struktur model: baseline ResNet50 + kelas blok ECA khusus varian (penyisipan eksplisit; lihat sel 'Penyisipan ECA').


In [ ]:
import os, sys, time, json, random, math, hashlib, shutil, platform, datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision, sklearn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, roc_auc_score)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# ---------------- KONFIGURASI ----------------
VARIANT = "v2_pre_add"
EXP_NAME = f"exp_{VARIANT}"
EXPECTED_PARAMS = 23_522_461        # baseline 23.522.375 + parameter ECA
EXPECTED_N_ECA = 16
SEEDS = [0, 1, 2]
CFG = dict(
    input_size=224, batch_size=10, lr=1e-4, epochs=100, num_classes=7,
    mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), num_workers=2,
    optimizer="Adam", split_seed=42,
)
SPLIT_CSV = "/kaggle/input/datasets/dinianjanip/xsdd-split/xsdd_split_seed42 (2).csv"
DATA_ROOT = "/kaggle/input/datasets/sayelabualigah/x-sdd/X-DDD"
SAVE_WEIGHTS = True                  # model_final.pt (~94 MB per seed), dipakai untuk Grad-CAM dan gerbang ECA

WORK_DIR = Path("/kaggle/working")
OUT_DIR = WORK_DIR / EXP_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
K = CFG["num_classes"]
BASELINE_PARAMS = 23_522_375

In [ ]:
# --- Data ---
split_df = pd.read_csv(SPLIT_CSV)
SPLIT_MD5 = hashlib.md5(Path(SPLIT_CSV).read_bytes()).hexdigest()
missing = [p for p in split_df.rel_path if not Path(DATA_ROOT, p).exists()]
assert not missing, f"{len(missing)} gambar tidak ditemukan, contoh: {missing[:3]}"

train_df = split_df[split_df.split == "train"].reset_index(drop=True)
test_df = split_df[split_df.split == "test"].reset_index(drop=True)
assert (len(train_df), len(test_df)) == (952, 408)
class_names = [n for _, n in sorted(set(zip(split_df.label_idx, split_df.label)))]
print("Kelas:", dict(enumerate(class_names)))
print("Train:", len(train_df), "| Test:", len(test_df), "| md5 split:", SPLIT_MD5)

tfm = transforms.Compose([
    transforms.Resize((CFG["input_size"], CFG["input_size"])),
    transforms.ToTensor(),
    transforms.Normalize(CFG["mean"], CFG["std"]),
])

class XSDD(Dataset):
    def __init__(self, df, root, transform):
        self.paths = [str(Path(root, p)) for p in df.rel_path]
        self.labels = df.label_idx.tolist()
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

train_ds = XSDD(train_df, DATA_ROOT, tfm)
test_ds = XSDD(test_df, DATA_ROOT, tfm)
Y_TRUE = np.array(test_ds.labels, dtype=np.int64)            # urutan = baris 'test' di CSV split

def make_loaders(seed):
    g = torch.Generator().manual_seed(seed)
    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, generator=g,
                              num_workers=CFG["num_workers"], pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False,
                             num_workers=CFG["num_workers"], pin_memory=pin)
    return train_loader, test_loader

## Penyisipan ECA (V2): setelah bn3, sebelum penjumlahan residual

Model baseline `ResNet50` dan modul `ECA` tidak diubah. ECA disisipkan lewat kelas blok `BottleneckECA_PreAdd` yang dipakai pada layer1 sampai layer4 oleh `ResNet50_ECA_V2`. Posisi penyisipan terlihat langsung di `forward` blok tersebut (baris bertanda `<<< ECA`) dan di daftar `blocks=(...)` pada `ResNet50_ECA_V2`.

```
x ──► conv1 1×1 ──► bn1 ──► ReLU ──► conv2 3×3 ──► bn2 ──► ReLU ──► conv3 1×1 ──► bn3 ──► [ ECA ] ──► (+) ──► ReLU ──► keluaran
│                                                                                                 ▲
└──────────────────────────────── identity (downsample jika perlu) ───────────────────────────────┘
```


In [ ]:
# --- Model: ResNet50 baseline + modul ECA, dengan titik penyisipan ECA yang eksplisit ---

# ============================================================
# 1) MODUL ECA (tidak diubah)
# ============================================================
class ECA(nn.Module):
    """ECA (Wang et al., 2020): GAP -> Conv1d lintas channel (kernel adaptif) -> sigmoid -> skala channel."""
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.record = False            # jika True, simpan nilai gerbang (B x C) di self.last_gate
        self.last_gate = None
    def forward(self, x):
        y = self.avg_pool(x)                              # B,C,1,1
        y = self.conv(y.squeeze(-1).transpose(-1, -2))    # B,1,C
        y = y.transpose(-1, -2).unsqueeze(-1)             # B,C,1,1
        gate = self.sigmoid(y)
        if self.record:
            self.last_gate = gate.detach().flatten(1)
        return x * gate

# ============================================================
# 2) BASELINE: Bottleneck dan ResNet50 TANPA ECA
# ============================================================
class Bottleneck(nn.Module):
    expansion = 4
    FLOW = [                                           # alur satu blok (untuk diagram blok)
        "x --> conv1 1x1 > bn1 > ReLU",
        "  --> conv2 3x3 > bn2 > ReLU",
        "  --> conv3 1x1 > bn3",
        "  --> (+) identity [downsample jika perlu] > ReLU --> keluaran blok",
    ]
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(planes * 4)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet50(nn.Module):
    """ResNet50 baseline. `blocks` = kelas blok pada layer1..layer4 (default: Bottleneck biasa, tanpa ECA)."""
    def __init__(self, num_classes=7, blocks=(Bottleneck, Bottleneck, Bottleneck, Bottleneck)):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_layer(blocks[0], 64, 3, stride=1)     # layer1: 3 blok,   64 ->  256 channel
        self.layer2 = self._make_layer(blocks[1], 128, 4, stride=2)    # layer2: 4 blok,  256 ->  512 channel
        self.layer3 = self._make_layer(blocks[2], 256, 6, stride=2)    # layer3: 6 blok,  512 -> 1024 channel
        self.layer4 = self._make_layer(blocks[3], 512, 3, stride=2)    # layer4: 3 blok, 1024 -> 2048 channel
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(2048, num_classes)
        for m in self.modules():                                       # inisialisasi Kaiming (from scratch)
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    def _make_layer(self, block, planes, n_blocks, stride):
        downsample = None
        if stride != 1 or self.inplanes != planes * 4:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * 4, 1, stride, bias=False),
                nn.BatchNorm2d(planes * 4),
            )
        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * 4
        layers += [block(self.inplanes, planes) for _ in range(1, n_blocks)]
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1))

# ============================================================
# 3) PENYISIPAN ECA (V2): setelah bn3, sebelum penjumlahan residual
# ============================================================
class BottleneckECA_PreAdd(Bottleneck):
    """Bottleneck dengan ECA: ECA setelah bn3 dan sebelum penjumlahan residual (posisi standar ECA-ResNet)."""
    FLOW = [
        "x --> conv1 1x1 > bn1 > ReLU",
        "  --> conv2 3x3 > bn2 > ReLU",
        "  --> conv3 1x1 > bn3 > [ECA]            <<< ECA DISISIPKAN DI SINI",
        "  --> (+) identity [downsample jika perlu] > ReLU --> keluaran blok",
    ]
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__(inplanes, planes, stride, downsample)
        self.eca = ECA(planes * 4)                   # modul ECA milik blok ini (dibuat setelah semua layer baseline)
    @property
    def eca_channels(self):
        return self.bn3.num_features
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        out = self.eca(out)                          # <<< ECA DISISIPKAN DI SINI: setelah bn3, sebelum penjumlahan residual
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet50_ECA_V2(ResNet50):
    """V2: ECA disisipkan di SEMUA blok (layer1-layer4), setelah bn3 dan sebelum penjumlahan residual. 16 modul, +86 parameter."""
    def __init__(self, num_classes=7):
        super().__init__(num_classes, blocks=(
            BottleneckECA_PreAdd,    # layer1: 3 blok -> 3 modul ECA (256 channel, kernel 5)    <<< ECA
            BottleneckECA_PreAdd,    # layer2: 4 blok -> 4 modul ECA (512 channel, kernel 5)    <<< ECA
            BottleneckECA_PreAdd,    # layer3: 6 blok -> 6 modul ECA (1024 channel, kernel 5)    <<< ECA
            BottleneckECA_PreAdd,    # layer4: 3 blok -> 3 modul ECA (2048 channel, kernel 7)    <<< ECA
        ))

# ============================================================
# 4) Pemilihan model dan ringkasan arsitektur
# ============================================================
MODEL_CLASSES = {
    "baseline": ResNet50,
    "v2_pre_add": ResNet50_ECA_V2,
}

def build_model(variant):
    return MODEL_CLASSES[variant](K)

def count_params(m):
    return sum(p.numel() for p in m.parameters())

def describe_architecture(model):
    """Ringkasan alur ResNet50 dan titik penyisipan ECA (untuk presentasi dan diagram blok).
    Catatan: print(model) menampilkan modul menurut urutan pendaftaran, bukan urutan eksekusi; gunakan ringkasan ini untuk alur."""
    print(f"{type(model).__name__}  (input 3x{CFG['input_size']}x{CFG['input_size']})")
    print("  Stem   : conv1 7x7/2 > bn1 > ReLU > maxpool 3x3/2")
    for name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, name)
        blk = layer[0]
        line = f"  {name} : {len(layer)} x {type(blk).__name__:<20s} {blk.conv1.in_channels:>4d} -> {blk.bn3.num_features:>4d} channel"
        if getattr(blk, "eca", None) is not None:
            line += f"   <<< ECA: {len(layer)} modul, {blk.eca_channels} channel, kernel {blk.eca.conv.kernel_size[0]}"
        print(line)
    print("  Head   : global avgpool > fc (7 kelas)")
    blk = model.layer1[0]
    print(f"\nAlur satu blok ({type(blk).__name__}):")
    for step in type(blk).FLOW:
        print("   ", step)

def complexity_report(model):
    """Parameter dan MACs per gambar 224x224 (Conv2d, Conv1d, Linear), plus operasi elemen-demi-elemen ECA."""
    model = model.cpu().eval()
    macs = {"conv2d": 0, "conv1d": 0, "linear": 0}
    extra = {"eca_elementwise_ops": 0}
    hooks = []
    def conv2d_hook(m, inp, out):
        macs["conv2d"] += out[0].numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
    def conv1d_hook(m, inp, out):
        macs["conv1d"] += out[0].numel() * (m.in_channels // m.groups) * m.kernel_size[0]
    def linear_hook(m, inp, out):
        macs["linear"] += m.in_features * m.out_features
    def eca_hook(m, inp, out):
        extra["eca_elementwise_ops"] += 2 * inp[0][0].numel()    # GAP (penjumlahan) + perkalian channel
    for m in model.modules():
        if isinstance(m, nn.Conv2d): hooks.append(m.register_forward_hook(conv2d_hook))
        elif isinstance(m, nn.Conv1d): hooks.append(m.register_forward_hook(conv1d_hook))
        elif isinstance(m, nn.Linear): hooks.append(m.register_forward_hook(linear_hook))
        if isinstance(m, ECA): hooks.append(m.register_forward_hook(eca_hook))
    with torch.no_grad():
        model(torch.zeros(1, 3, CFG["input_size"], CFG["input_size"]))
    for h in hooks:
        h.remove()
    ecas = [(n, m.conv.kernel_size[0]) for n, m in model.named_modules() if isinstance(m, ECA)]
    return dict(
        variant=VARIANT, n_params=count_params(model),
        n_params_trainable=sum(p.numel() for p in model.parameters() if p.requires_grad),
        n_eca_modules=len(ecas), eca_kernel_sizes={n: k for n, k in ecas},
        macs_conv2d=macs["conv2d"], macs_conv1d=macs["conv1d"], macs_linear=macs["linear"],
        macs_total=macs["conv2d"] + macs["conv1d"] + macs["linear"],
        eca_elementwise_ops=extra["eca_elementwise_ops"],
        input_shape=[1, 3, CFG["input_size"], CFG["input_size"]],
    )

# --- Pemeriksaan model sebelum training ---
describe_architecture(build_model(VARIANT))      # ringkasan alur dan titik penyisipan ECA
print()
COMPLEXITY = complexity_report(build_model(VARIANT))
print(json.dumps({k: v for k, v in COMPLEXITY.items() if k != "eca_kernel_sizes"}, indent=1))
assert COMPLEXITY["n_params"] == EXPECTED_PARAMS, f"parameter {COMPLEXITY['n_params']:,} != harapan {EXPECTED_PARAMS:,}"
assert COMPLEXITY["n_eca_modules"] == EXPECTED_N_ECA, f"modul ECA {COMPLEXITY['n_eca_modules']} != harapan {EXPECTED_N_ECA}"
_m = build_model(VARIANT).eval()
with torch.no_grad():
    assert _m(torch.randn(2, 3, 224, 224)).shape == (2, K)
del _m
print("OK: model valid.")
with open(OUT_DIR / "complexity.json", "w") as f:
    json.dump(COMPLEXITY, f, indent=1)

In [ ]:
# --- Metadata keterulangan & README ---
config = dict(
    variant=VARIANT, exp_name=EXP_NAME, seeds=SEEDS, cfg=CFG, expected_params=EXPECTED_PARAMS,
    split_csv=SPLIT_CSV, split_md5=SPLIT_MD5, data_root=DATA_ROOT, class_names=class_names,
    protocol="from scratch, tanpa augmentasi, tanpa early stopping, bobot epoch 100",
    libs=dict(python=platform.python_version(), torch=torch.__version__, torchvision=torchvision.__version__,
              sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__),
    cuda=dict(torch_cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
              gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"),
    started_at=datetime.datetime.now().isoformat(timespec="seconds"),
)
with open(OUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=1)

README = """Isi folder eksperimen
config.json            hyperparameter, seed, md5 file split, versi library, GPU
complexity.json        jumlah parameter, MACs per gambar 224x224, operasi elemen ECA
summary_seeds.csv      metrik utama per seed (ditulis di akhir)
seedN/predictions.csv  rel_path, y_true, y_pred, correct, logit_0..6, prob_0..6 (urutan = baris 'test' di CSV split)
seedN/metrics_overall.json   accuracy, macro-P/R/F1, loss(NLL), Brier, AUC, rata-rata epoch 91-100, waktu
seedN/metrics_per_class.csv  precision, recall, f1, support per kelas
seedN/confusion_matrix.csv   baris = kelas benar, kolom = kelas prediksi
seedN/history.csv      per epoch: train/test loss dan metrik, waktu epoch, memori GPU puncak (training)
seedN/model_final.pt   state_dict epoch 100 (jika SAVE_WEIGHTS)
seedN/gates_test.npz   nilai gerbang ECA tiap gambar uji per modul (float16), hanya varian ECA
seedN/gates_summary.json   statistik gerbang per modul (rata-rata, std, eta^2 antar kelas)
aggregate/summary_mean_std.csv        rata-rata, std (sampel, ddof=1), min, max antar seed untuk metrik keseluruhan
aggregate/per_class_mean_std.csv      precision, recall, F1 per kelas: rata-rata, std, min, max, dan nilai tiap seed
aggregate/confusion_matrix_sum.csv    confusion matrix dijumlahkan antar seed (juga _mean dan _row_normalized)
aggregate/seed_agreement.csv          tiap gambar uji: prediksi dan benar/salah per seed, proporsi seed yang benar
Catatan: uji statistik tingkat run memakai data per seed (seedN/), bukan agregat.
aggregate/             agregasi antar seed: mean/std metrik keseluruhan dan per kelas, confusion matrix (jumlah, rata-rata,
                       dinormalisasi), prediksi per gambar digabung, riwayat mean/std (lihat README_aggregate.txt di dalamnya)
"""
(OUT_DIR / "README.txt").write_text(README)

In [ ]:
# --- Fungsi training dan evaluasi ---
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    outs = []
    for x, _ in loader:
        outs.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(outs).numpy()

def lite_metrics(y_true, logits):
    loss = F.cross_entropy(torch.from_numpy(logits), torch.from_numpy(y_true)).item()
    y_pred = logits.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return dict(loss=float(loss), acc=float(accuracy_score(y_true, y_pred)),
                macro_precision=float(p), macro_recall=float(r), macro_f1=float(f1))

def full_metrics(y_true, logits):
    m = lite_metrics(y_true, logits)
    prob = torch.softmax(torch.from_numpy(logits).double(), 1).numpy()
    prob = prob / prob.sum(1, keepdims=True)
    onehot = np.eye(K)[y_true]
    m["brier"] = float(((prob - onehot) ** 2).sum(1).mean())
    try:
        m["macro_auc_ovr"] = float(roc_auc_score(y_true, prob, multi_class="ovr", average="macro", labels=list(range(K))))
    except ValueError:
        m["macro_auc_ovr"] = float("nan")
    return m, logits.argmax(1), prob

@torch.no_grad()
def collect_gates(model, loader):
    ecas = [(n, m) for n, m in model.named_modules() if isinstance(m, ECA)]
    if not ecas:
        return None
    for _, m in ecas:
        m.record = True
        m._buf = []
    model.eval()
    for x, _ in loader:
        model(x.to(DEVICE, non_blocking=True))
        for _, m in ecas:
            m._buf.append(m.last_gate.float().cpu())
    gates = {}
    for n, m in ecas:
        gates[n] = torch.cat(m._buf).numpy()               # N x C
        m.record, m._buf, m.last_gate = False, [], None
    return gates

def gate_summary(gates, y_true):
    out = {}
    classes = np.unique(y_true)
    for name, G in gates.items():
        G = G.astype(np.float64)
        mu = G.mean(0)
        sst = ((G - mu) ** 2).sum(0)
        ssb = sum((y_true == k).sum() * (G[y_true == k].mean(0) - mu) ** 2 for k in classes)
        eta2 = np.where(sst > 1e-12, ssb / np.maximum(sst, 1e-12), 0.0)
        out[name] = dict(channels=int(G.shape[1]), mean=float(G.mean()), std_across_images_mean=float(G.std(0).mean()),
                         min=float(G.min()), max=float(G.max()), eta2_between_class_mean=float(eta2.mean()))
    return out

def run_seed(seed):
    seed_dir = OUT_DIR / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    set_seed(seed)
    train_loader, test_loader = make_loaders(seed)
    model = build_model(VARIANT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
    history = []
    wall0 = time.time()
    for ep in range(1, CFG["epochs"] + 1):
        model.train()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        t0 = time.time()
        loss_sum, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            out = model(x)
            loss = F.cross_entropy(out, y)
            loss.backward()
            opt.step()
            loss_sum += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_time = time.time() - t0
        peak_mb = torch.cuda.max_memory_allocated() / 2**20 if DEVICE.type == "cuda" else 0.0
        m = lite_metrics(Y_TRUE, predict_logits(model, test_loader))     # diagnostik, tidak dipakai memilih epoch
        rec = dict(epoch=ep, train_loss=loss_sum / n, train_acc=correct / n, epoch_time_s=epoch_time,
                   peak_mem_mb=peak_mb, **{f"test_{k}": v for k, v in m.items()})
        history.append(rec)
        pd.DataFrame(history).to_csv(seed_dir / "history.csv", index=False)   # aman jika sesi terputus
        if ep == 1 or ep % 10 == 0:
            print(f"[{VARIANT} seed{seed}] ep {ep:3d} | train_loss {rec['train_loss']:.4f} acc {rec['train_acc']:.3f} "
                  f"| test_acc {rec['test_acc']:.3f} F1 {rec['test_macro_f1']:.3f} | {epoch_time:.1f}s")

    # ---- Evaluasi akhir = model epoch 100 ----
    logits = predict_logits(model, test_loader)
    final, y_pred, prob = full_metrics(Y_TRUE, logits)

    pred_df = pd.DataFrame({"rel_path": test_df.rel_path.values, "y_true": Y_TRUE, "y_pred": y_pred,
                            "correct": (Y_TRUE == y_pred).astype(int)})
    for k in range(K):
        pred_df[f"logit_{k}"] = logits[:, k]
    for k in range(K):
        pred_df[f"prob_{k}"] = prob[:, k]
    pred_df.to_csv(seed_dir / "predictions.csv", index=False)

    report = classification_report(Y_TRUE, y_pred, labels=list(range(K)), target_names=class_names,
                                   output_dict=True, zero_division=0)
    pd.DataFrame(report).T.to_csv(seed_dir / "metrics_per_class.csv")
    cm = confusion_matrix(Y_TRUE, y_pred, labels=list(range(K)))
    pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                 columns=[f"pred_{c}" for c in class_names]).to_csv(seed_dir / "confusion_matrix.csv")

    hist = pd.DataFrame(history)
    last10 = hist[hist.epoch >= CFG["epochs"] - 9]
    times = hist.epoch_time_s.values
    meta = dict(
        variant=VARIANT, seed=seed, n_params=COMPLEXITY["n_params"], final=final,
        last10_mean={k: float(last10[f"test_{k}"].mean()) for k in ["loss", "acc", "macro_precision", "macro_recall", "macro_f1"]},
        timing=dict(train_time_total_s=float(times.sum()), epoch_time_mean_s_excl_first=float(times[1:].mean()),
                    wall_time_s=float(time.time() - wall0), peak_mem_mb_max=float(hist.peak_mem_mb.max())),
    )
    with open(seed_dir / "metrics_overall.json", "w") as f:
        json.dump(meta, f, indent=1)

    gates = collect_gates(model, test_loader)
    if gates is not None:
        np.savez_compressed(seed_dir / "gates_test.npz", y_true=Y_TRUE,
                            **{k: v.astype(np.float16) for k, v in gates.items()})
        with open(seed_dir / "gates_summary.json", "w") as f:
            json.dump(gate_summary(gates, Y_TRUE), f, indent=1)
    if SAVE_WEIGHTS:
        torch.save(model.state_dict(), seed_dir / "model_final.pt")
    (seed_dir / "DONE").write_text("ok")

    print(f"\n==> {VARIANT} seed {seed} SELESAI | acc {final['acc']:.4f} | macroP {final['macro_precision']:.4f} | "
          f"macroR {final['macro_recall']:.4f} | macroF1 {final['macro_f1']:.4f} | NLL {final['loss']:.4f} | "
          f"Brier {final['brier']:.4f} | AUC {final['macro_auc_ovr']:.4f} | waktu train {times.sum()/60:.1f} menit\n")
    del model, opt
    torch.cuda.empty_cache()

In [ ]:
# --- Jalankan semua seed (seed yang sudah selesai dilewati) ---
for seed in SEEDS:
    if (OUT_DIR / f"seed{seed}" / "DONE").exists():
        print(f"seed {seed}: sudah selesai, dilewati")
        continue
    run_seed(seed)

## Ringkasan, pemeriksaan integritas, agregasi antar seed, dan zip

In [ ]:
rows = []
for seed in SEEDS:
    meta = json.load(open(OUT_DIR / f"seed{seed}" / "metrics_overall.json"))
    f_, l_, t_ = meta["final"], meta["last10_mean"], meta["timing"]
    rows.append(dict(seed=seed, acc=f_["acc"], macro_precision=f_["macro_precision"], macro_recall=f_["macro_recall"],
                     macro_f1=f_["macro_f1"], nll=f_["loss"], brier=f_["brier"], macro_auc_ovr=f_["macro_auc_ovr"],
                     acc_last10=l_["acc"], macro_f1_last10=l_["macro_f1"],
                     train_time_min=t_["train_time_total_s"] / 60, epoch_time_s=t_["epoch_time_mean_s_excl_first"],
                     peak_mem_mb=t_["peak_mem_mb_max"], n_params=meta["n_params"]))
summary = pd.DataFrame(rows).set_index("seed")
summary.to_csv(OUT_DIR / "summary_seeds.csv")
print(summary.round(4).to_string())
print("\nmean ± std (std sampel):")
print(pd.DataFrame({"mean": summary.mean(), "std": summary.std(ddof=1)}).round(4).to_string())

# --- Pemeriksaan integritas: semua berkas ada, dan metrik cocok dengan prediksi per gambar ---
REQUIRED = ["predictions.csv", "metrics_overall.json", "metrics_per_class.csv", "confusion_matrix.csv", "history.csv", "DONE"]
if SAVE_WEIGHTS: REQUIRED.append("model_final.pt")
if EXPECTED_N_ECA > 0: REQUIRED += ["gates_test.npz", "gates_summary.json"]
for seed in SEEDS:
    d = OUT_DIR / f"seed{seed}"
    absent = [f for f in REQUIRED if not (d / f).exists()]
    assert not absent, f"seed {seed}: berkas hilang: {absent}"
    pr = pd.read_csv(d / "predictions.csv")
    assert len(pr) == 408 and (pr.y_true.values == Y_TRUE).all()
    meta = json.load(open(d / "metrics_overall.json"))
    f1_re = precision_recall_fscore_support(pr.y_true, pr.y_pred, average="macro", zero_division=0)[2]
    assert abs((pr.y_true == pr.y_pred).mean() - meta["final"]["acc"]) < 1e-9
    assert abs(f1_re - meta["final"]["macro_f1"]) < 1e-9
print("\nOK: semua berkas lengkap dan metrik cocok dengan prediksi per gambar.")

# --- Agregasi antar seed (folder aggregate) ---
def aggregate_experiment(exp_dir, out_dir=None):
    """Agregasi antar seed untuk satu folder eksperimen: ringkasan mean/std, confusion matrix, prediksi, riwayat."""
    exp_dir = Path(exp_dir)
    seed_dirs = sorted([p for p in exp_dir.glob("seed*") if (p / "metrics_overall.json").exists()],
                       key=lambda p: int(p.name.replace("seed", "")))
    assert len(seed_dirs) >= 1, "tidak ada folder seed yang lengkap"
    names = [p.name for p in seed_dirs]
    out = Path(out_dir) if out_dir else exp_dir / "aggregate"
    out.mkdir(parents=True, exist_ok=True)

    # 1) Metrik keseluruhan: mean, std (sampel), min, max, dan nilai tiap seed
    cols = {}
    for sd in seed_dirs:
        meta = json.load(open(sd / "metrics_overall.json"))
        flat = dict(meta["final"])
        flat.update({f"{k}_last10": v for k, v in meta["last10_mean"].items()})
        flat.update({f"time_{k}": v for k, v in meta["timing"].items()})
        cols[sd.name] = flat
    df = pd.DataFrame(cols).rename(index={"loss": "nll", "loss_last10": "test_loss_last10"})
    df.index.name = "metric"
    overall = pd.DataFrame({"mean": df.mean(axis=1), "std": df.std(axis=1, ddof=1),
                            "min": df.min(axis=1), "max": df.max(axis=1)})
    overall["n_seeds"] = len(seed_dirs)
    overall = pd.concat([overall, df], axis=1)
    overall.to_csv(out / "metrics_overall_mean_std.csv")

    # 2) Metrik per kelas: mean, std, min, max, dan nilai tiap seed
    pcs = [pd.read_csv(sd / "metrics_per_class.csv", index_col=0).drop(index="accuracy", errors="ignore")
           for sd in seed_dirs]
    mets = ["precision", "recall", "f1-score"]
    arr = np.stack([p[mets].values for p in pcs])                  # seed x baris x metrik
    recs = []
    for i, cname in enumerate(pcs[0].index):
        for j, met in enumerate(mets):
            v = arr[:, i, j]
            rec = dict(class_name=cname, metric=met, mean=v.mean(),
                       std=v.std(ddof=1) if len(v) > 1 else np.nan, min=v.min(), max=v.max(),
                       support=pcs[0].loc[cname, "support"])
            rec.update({n: x for n, x in zip(names, v)})
            recs.append(rec)
    pd.DataFrame(recs).to_csv(out / "metrics_per_class_mean_std.csv", index=False)

    # 3) Confusion matrix: jumlah, rata-rata, dan rata-rata yang dinormalisasi per baris (= recall antar kelas)
    cms = [pd.read_csv(sd / "confusion_matrix.csv", index_col=0) for sd in seed_dirs]
    cm_sum = sum(cms)
    (cm_sum).to_csv(out / "confusion_matrix_sum.csv")
    (cm_sum / len(cms)).to_csv(out / "confusion_matrix_mean.csv")
    (sum(c.div(c.sum(axis=1), axis=0) for c in cms) / len(cms)).to_csv(out / "confusion_matrix_rownorm_mean.csv")

    # 4) Prediksi per gambar uji, digabung antar seed
    preds = [pd.read_csv(sd / "predictions.csv") for sd in seed_dirs]
    agg = preds[0][["rel_path", "y_true"]].copy()
    for p in preds[1:]:
        assert (p.rel_path.values == agg.rel_path.values).all() and (p.y_true.values == agg.y_true.values).all()
    K = len([c for c in preds[0].columns if c.startswith("prob_")])
    for n, p in zip(names, preds):
        agg[f"pred_{n}"] = p.y_pred.values
        agg[f"correct_{n}"] = p.correct.values
    agg["n_correct"] = agg[[f"correct_{n}" for n in names]].sum(axis=1)
    agg["all_seeds_agree"] = (agg[[f"pred_{n}" for n in names]].nunique(axis=1) == 1).astype(int)
    for k in range(K):
        agg[f"prob_mean_{k}"] = np.mean([p[f"prob_{k}"].values for p in preds], axis=0)
    for k in range(K):
        agg[f"logit_mean_{k}"] = np.mean([p[f"logit_{k}"].values for p in preds], axis=0)
    agg["ensemble_pred"] = agg[[f"prob_mean_{k}" for k in range(K)]].values.argmax(1)
    agg["ensemble_correct"] = (agg["ensemble_pred"] == agg["y_true"]).astype(int)
    agg.to_csv(out / "predictions_aggregate.csv", index=False)

    # 5) Riwayat per epoch: mean dan std antar seed
    hs = [pd.read_csv(sd / "history.csv") for sd in seed_dirs]
    hc = pd.concat(hs, keys=names, names=["seed"]).reset_index(level=0)
    num = [c for c in hs[0].columns if c != "epoch"]
    g = hc.groupby("epoch")[num]
    hist = pd.concat([g.mean().add_suffix("_mean"), g.std(ddof=1).add_suffix("_std")], axis=1).reset_index()
    hist.to_csv(out / "history_mean_std.csv", index=False)

    (out / "README_aggregate.txt").write_text(
        "Agregasi antar seed (folder aggregate)\n"
        "metrics_overall_mean_std.csv      mean, std (sampel), min, max, dan nilai tiap seed untuk metrik keseluruhan\n"
        "metrics_per_class_mean_std.csv    precision/recall/f1 per kelas: mean, std, min, max, nilai tiap seed, support\n"
        "confusion_matrix_sum.csv          jumlah confusion matrix semua seed (baris = kelas benar, kolom = prediksi)\n"
        "confusion_matrix_mean.csv         rata-rata jumlah per sel antar seed\n"
        "confusion_matrix_rownorm_mean.csv rata-rata proporsi per baris (diagonal = recall per kelas)\n"
        "predictions_aggregate.csv         per gambar uji: prediksi dan benar/salah tiap seed, n_correct (berapa seed benar),\n"
        "                                  all_seeds_agree, rata-rata probabilitas dan logit antar seed,\n"
        "                                  ensemble_pred/ensemble_correct (argmax rata-rata probabilitas)\n"
        "                                  CATATAN: ensemble_* adalah gabungan seed, BUKAN hasil satu model; jangan dibandingkan\n"
        "                                  dengan metrik per seed. Uji McNemar dan bootstrap memakai prediksi per seed.\n"
        "history_mean_std.csv              riwayat per epoch: mean dan std antar seed\n")
    return dict(n_seeds=len(seed_dirs), overall=overall, files=sorted(p.name for p in out.iterdir()))

agg_res = aggregate_experiment(OUT_DIR)
NEED_AGG = ["metrics_overall_mean_std.csv", "metrics_per_class_mean_std.csv", "confusion_matrix_sum.csv",
            "confusion_matrix_mean.csv", "confusion_matrix_rownorm_mean.csv", "predictions_aggregate.csv", "history_mean_std.csv"]
assert all((OUT_DIR / "aggregate" / f).exists() for f in NEED_AGG), "berkas agregasi tidak lengkap"
print("\nOK: folder aggregate lengkap:", agg_res["files"])

## Agregasi antar seed (folder `aggregate/`)
Turunan untuk tabel dan grafik di skripsi. Uji statistik tingkat run tetap memakai data per seed (`seedN/`).
Prediksi TIDAK dirata-ratakan menjadi satu model (itu akan menjadi ensemble); yang disimpan adalah kesepakatan antar seed.

In [ ]:
AGG_DIR = OUT_DIR / "aggregate"
AGG_DIR.mkdir(exist_ok=True)

# 1) Metrik keseluruhan: rata-rata, std sampel (ddof=1), min, max antar seed
summary_mean_std = pd.DataFrame({"mean": summary.mean(), "std": summary.std(ddof=1),
                                 "min": summary.min(), "max": summary.max()})
summary_mean_std.to_csv(AGG_DIR / "summary_mean_std.csv")

# 2) Metrik per kelas (precision, recall, F1) antar seed
pcs = [pd.read_csv(OUT_DIR / f"seed{s}" / "metrics_per_class.csv", index_col=0) for s in SEEDS]
rows = []
for cls in class_names + ["macro avg"]:
    support = float(pcs[0].loc[cls, "support"])
    for metric in ["precision", "recall", "f1-score"]:
        vals = np.array([pc.loc[cls, metric] for pc in pcs], dtype=float)
        row = dict(kelas=cls, metrik=metric, support=support, mean=vals.mean(), std=vals.std(ddof=1),
                   min=vals.min(), max=vals.max())
        row.update({f"seed{s}": v for s, v in zip(SEEDS, vals)})
        rows.append(row)
per_class_ms = pd.DataFrame(rows)
per_class_ms.to_csv(AGG_DIR / "per_class_mean_std.csv", index=False)

# 3) Confusion matrix: jumlah, rata-rata, dan dinormalisasi per baris (kelas benar)
cms = [pd.read_csv(OUT_DIR / f"seed{s}" / "confusion_matrix.csv", index_col=0) for s in SEEDS]
cm_sum = sum(cms)
cm_sum.to_csv(AGG_DIR / "confusion_matrix_sum.csv")
(cm_sum / len(SEEDS)).round(4).to_csv(AGG_DIR / "confusion_matrix_mean.csv")
cm_sum.div(cm_sum.sum(axis=1), axis=0).round(4).to_csv(AGG_DIR / "confusion_matrix_row_normalized.csv")

# 4) Kesepakatan antar seed per gambar uji (bukan ensemble)
agree = None
for s in SEEDS:
    p = pd.read_csv(OUT_DIR / f"seed{s}" / "predictions.csv")
    if agree is None:
        agree = p[["rel_path", "y_true"]].copy()
    assert (p.rel_path.values == agree.rel_path.values).all() and (p.y_true.values == agree.y_true.values).all()
    agree[f"y_pred_seed{s}"] = p.y_pred.values
    agree[f"correct_seed{s}"] = p.correct.values
correct_cols = [f"correct_seed{s}" for s in SEEDS]
pred_cols = [f"y_pred_seed{s}" for s in SEEDS]
agree["n_correct"] = agree[correct_cols].sum(axis=1)
agree["prop_correct"] = agree["n_correct"] / len(SEEDS)
agree["pred_consistent"] = agree[pred_cols].nunique(axis=1) == 1
agree.to_csv(AGG_DIR / "seed_agreement.csv", index=False)

# --- Pemeriksaan agregat ---
assert abs(summary_mean_std.loc["acc", "mean"] - summary["acc"].mean()) < 1e-12
assert int(cm_sum.values.sum()) == 408 * len(SEEDS)
for f in ["summary_mean_std.csv", "per_class_mean_std.csv", "confusion_matrix_sum.csv",
          "confusion_matrix_mean.csv", "confusion_matrix_row_normalized.csv", "seed_agreement.csv"]:
    assert (AGG_DIR / f).exists(), f"berkas agregat hilang: {f}"

print("Metrik keseluruhan (mean ± std antar seed):")
print(summary_mean_std.loc[["acc", "macro_precision", "macro_recall", "macro_f1", "macro_f1_last10"], ["mean", "std"]].round(4).to_string())
print("\nMetrik per kelas (rata-rata antar seed):")
print(per_class_ms.pivot(index="kelas", columns="metrik", values="mean").round(3).to_string())
print("\nConfusion matrix (jumlah antar seed):")
print(cm_sum.to_string())
print("\nKesepakatan antar seed: jumlah gambar uji menurut banyaknya seed yang benar")
print(agree["n_correct"].value_counts().sort_index().to_string())
print("Prediksi identik di semua seed:", int(agree["pred_consistent"].sum()), "dari", len(agree), "gambar")
print("OK: berkas agregat lengkap di", AGG_DIR)

In [ ]:
# --- Zip untuk diunduh dari tab Output ---
zip_path = shutil.make_archive(str(WORK_DIR / EXP_NAME), "zip", root_dir=str(WORK_DIR), base_dir=EXP_NAME)
print("ZIP:", zip_path, f"({os.path.getsize(zip_path) / 2**20:.0f} MB)")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(WORK_DIR)):55s} {p.stat().st_size / 1024:10.1f} KB")